# Notebook 5: Probability for RL (expectation, noise, and why averages behave)

Step 5 of the ladder. Notebooks 1 to 4 were about *learning a function* from data. Reinforcement learning adds something new: **the data itself is random**. The same action in the same state can give different rewards, and a policy doesn't choose an action, it chooses a **probability distribution over actions**. Words like *return*, *expected value*, *advantage*, *entropy* and *seed* all come from probability. This notebook makes each one something you can compute and check.

You need nothing from earlier notebooks except the "nudge and measure" habit. NumPy and matplotlib only; runs in seconds.

### What you'll be able to say at the end
- A **distribution** is a table of outcomes and their probabilities; a **policy** is one.
- **Expected value** is the probability-weighted average, and a Q-value *is* one.
- Averages of random things settle down, but **slowly** (the error shrinks like 1 over the square root of the number of samples). That one fact explains why RL needs so many episodes.
- Why subtracting a **baseline** makes policy-gradient learning much less noisy without changing where it's headed, and why the baseline must not depend on the action taken.
- What **entropy** measures, why **gamma** can be read as a survival chance, and how many **seeds** you need before you can trust "A beat B".

Same markers as before: `WHAT / WHY / SAME AS / BECOMES / WATCH` comments, **PREDICT** cells, a knob ledger, and a sandbox.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

# WHAT: plot styling only (same quiet look as earlier notebooks). Nothing here affects the math.
BLUE, ORANGE, AQUA, YELLOW = "#2a78d6", "#eb6834", "#1baf7a", "#eda100"
INK, MUTED, GRIDC = "#0b0b0b", "#52514e", "#e1e0d9"
plt.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb", "savefig.facecolor": "#fcfcfb",
    "axes.edgecolor": "#c3c2b7", "axes.labelcolor": MUTED, "xtick.color": MUTED, "ytick.color": MUTED,
    "text.color": INK, "axes.titlecolor": INK, "axes.titlesize": 11, "axes.labelsize": 10, "font.size": 10,
    "axes.grid": True, "grid.color": GRIDC, "grid.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False,
    "lines.linewidth": 2.0, "legend.frameon": False, "legend.fontsize": 9, "figure.dpi": 110,
})

def tag(ax, x, y, text, color, dx=4, dy=0, ha="left"):
    # WHAT: write a small label next to a line instead of making you decode a legend.
    ax.annotate(text, (x, y), xytext=(dx, dy), textcoords="offset points", color=color, fontsize=9, va="center", ha=ha)

print("ready")

ready


# Part 1: A distribution, and a policy is one

> **You are here:** the basic object. A **distribution** lists the possible outcomes and how likely each is. The probabilities are non-negative and add to 1.

A RL **policy**, for one state, is exactly this: a list of actions with a probability on each. "Push left 50%, push right 30%, do nothing 20%" is a policy. To *act*, you **sample** from it: draw one action at random according to those probabilities. Do that many times and the fraction of each action approaches its probability. Watch it:

In [2]:
actions = ["left", "right", "stay"]
policy = np.array([0.5, 0.3, 0.2])                    # WHAT: the distribution: one probability per action. They must add to 1.
print("probabilities add to:", policy.sum())

rng = np.random.default_rng(0)                        # WHAT: a seeded random number generator. Same seed = same "random" draws (Part 8 returns to this).
print()
print("  samples |  left  |  right |  stay   (fraction of draws)")
for n in (10, 100, 10_000):
    draws = rng.choice(len(actions), size=n, p=policy)          # WHAT: sample n actions according to the policy
    frac = np.bincount(draws, minlength=3) / n
    print(f" {n:8d} | {frac[0]:6.3f} | {frac[1]:6.3f} | {frac[2]:6.3f}")
print(f"    true   | {policy[0]:6.3f} | {policy[1]:6.3f} | {policy[2]:6.3f}")

probabilities add to: 1.0

  samples |  left  |  right |  stay   (fraction of draws)
       10 |  0.300 |  0.400 |  0.300
      100 |  0.460 |  0.270 |  0.270
    10000 |  0.500 |  0.299 |  0.201
    true   |  0.500 |  0.300 |  0.200


**What to notice.** With 10 draws the fractions are rough (one unlucky draw moves a fraction by 10 points). With 10,000 they sit within about a percentage point of the truth. Nothing is wrong with the generator. **Small samples are just noisy.** Keep that in mind: an RL agent that has taken 10 actions has a very rough idea of what those actions do.

(In the earlier notebooks, `epsilon` exploration was a tiny policy of this kind: "best action with probability 1-epsilon, otherwise a random one". REINFORCE and A2C learn the probabilities themselves.)

# Part 2: Expected value and variance

> **You are here:** a single outcome is random. Is there a single *number* that summarizes what to expect?

**Expected value** is each outcome times its probability, added up: `E[X] = sum( p * x )`. It's the long-run average. Example, a machinist's version: each part you cut ends in one of three ways, with a cost to you:

| Outcome | Probability | Cost |
|---|---|---|
| passes first time | 0.90 | $0 |
| needs rework | 0.07 | $15 |
| scrapped | 0.03 | $80 |

In [3]:
costs = np.array([0.0, 15.0, 80.0])
probs = np.array([0.90, 0.07, 0.03])

expected = np.sum(probs * costs)                           # WHAT: E[X] = sum of (probability x value)
second = np.sum(probs * costs ** 2)                        # WHAT: E[X^2]: the same sum, with each value squared
variance = second - expected ** 2                          # WHAT: variance = E[X^2] - (E[X])^2 = the average squared distance from the mean
std = np.sqrt(variance)                                    # WHAT: standard deviation: variance back in the original units ($)
print(f"expected cost per part : ${expected:.2f}")
print(f"variance               : {variance:.1f}  ($ squared)")
print(f"standard deviation     : ${std:.2f}")
print()

rng = np.random.default_rng(1)
sim = rng.choice(costs, size=200_000, p=probs)             # WHAT: simulate 200,000 parts and just average what happened
print(f"simulated average of 200,000 parts: ${sim.mean():.3f}   (formula said ${expected:.3f})")
print(f"fraction of parts that cost exactly the 'expected' ${expected:.2f}: {np.mean(sim == expected):.0%}")

expected cost per part : $3.45
variance               : 195.8  ($ squared)
standard deviation     : $13.99

simulated average of 200,000 parts: $3.420   (formula said $3.450)
fraction of parts that cost exactly the 'expected' $3.45: 0%


**What to notice.**
- The **expected cost is $3.45, but no single part ever costs $3.45**: you'll see $0, $15 or $80. Expected value is a long-run average, not a prediction of any one event.
- The standard deviation ($14.0) is **four times** the mean ($3.45). One part tells you almost nothing about the mean.
- **A Q-value is exactly an expected value:** `Q(s, a) = E[ return | start in s, take a ]`. Q-learning's table entry is trying to learn this number from random samples.
- **Variance** measures how much single outcomes scatter around the mean. High variance = noisy samples = many samples needed (next part).

# Part 3: Averages settle down, but slowly (the square-root law)

> **You are here:** you can average samples to estimate an expectation. How good is that estimate after `n` samples?

**PREDICT first.** For the cost data above, suppose you average `n` parts. If you want the estimate **10 times more precise** (error 10 times smaller), roughly how many *times more* parts do you need: 10x, 100x, or 1000x? Type a guess, then run.

In [4]:
# PREDICT: to cut the error of an average by 10x, you need about ___ times more samples.
my_prediction = ""
print("your prediction:", my_prediction if my_prediction else "(blank: type one above first)")

your prediction: (blank: type one above first)


In [5]:
rng = np.random.default_rng(2)
print("   n    | spread of the sample mean (std over 2000 repeats) | formula: sigma / sqrt(n)")
spreads = {}
for n in (1, 10, 100, 1000, 10_000):
    means = rng.choice(costs, size=(2000, n), p=probs).mean(axis=1)    # WHAT: 2000 independent experiments, each averages n parts
    spreads[n] = means.std()                                           # WHAT: how much do those 2000 averages disagree with each other?
    print(f" {n:6d} |                 {spreads[n]:7.3f}                    |        {std / np.sqrt(n):7.3f}")
print()
print(f"error at n=100 divided by error at n=10,000: {spreads[100] / spreads[10_000]:.1f}   (100x the data bought a 10x smaller error)")

   n    | spread of the sample mean (std over 2000 repeats) | formula: sigma / sqrt(n)
      1 |                  13.918                    |         13.995
     10 |                   4.331                    |          4.425
    100 |                   1.371                    |          1.399
   1000 |                   0.452                    |          0.443


  10000 |                   0.143                    |          0.140

error at n=100 divided by error at n=10,000: 9.6   (100x the data bought a 10x smaller error)


In [6]:
rng = np.random.default_rng(3)
N = 5000
fig, ax = plt.subplots(figsize=(7.6, 4.1))
ns = np.arange(1, N + 1)
for _ in range(12):
    ax.plot(ns, np.cumsum(rng.choice(costs, size=N, p=probs)) / ns, color=BLUE, alpha=0.5, linewidth=1.2)   # running average of one experiment
ax.plot(ns, expected + 2 * std / np.sqrt(ns), color=ORANGE, linestyle="--", linewidth=1.5)
ax.plot(ns, expected - 2 * std / np.sqrt(ns), color=ORANGE, linestyle="--", linewidth=1.5)
ax.axhline(expected, color=INK, linewidth=1.2)
ax.set_xscale("log"); ax.set_ylim(-2, 20); ax.set_xlabel("number of parts averaged (log scale)"); ax.set_ylabel("running average cost ($)")
tag(ax, 6500, expected, "true mean $3.45", INK, dx=0); tag(ax, 6500, 7.0, "mean +/- 2 sigma/sqrt(n)", ORANGE, dx=0)
ax.set_xlim(1, 60000); ax.set_title("12 separate experiments (blue) settle into a narrowing funnel (orange)")
plt.show()

<Figure size 836x451 with 1 Axes>

**What to notice.**
- **100 times more data bought only a 10 times smaller error.** The spread of an average shrinks like `sigma / sqrt(n)`: to halve the error you need **4 times** the samples, and for 10 times better you need **100 times**. (The table's measured column matches the formula, so this isn't an approximation I'm asserting.)
- The blue lines start all over the place and funnel in toward the true mean. That funnel is the **law of large numbers**: averages converge to expectations, but at the slow square-root rate.
- This is *why RL needs so many episodes*. One episode's return is one noisy sample, like one part. It's also why the **batch size** knob in Notebook 1 mattered: bigger batches average more samples per step, so the slope is less noisy.

# Part 4: Transitions from experience, and the first value-estimation method

> **You are here:** the world's randomness in RL has a name: **transition probabilities**, `P(next state | state, action)`. Here is a tiny slippery hallway you can compute exactly, then estimate from experience.

The hallway has cells 0 to 4. Cell 4 is the goal (the episode ends). The policy always says "right". But the floor is slippery: **with probability 0.8 you move right, with 0.2 you stay put.** Each step costs -0.04; arriving at the goal gives +1. Gamma = 0.9. (Same reward style as the Qmaze notebook.)

**Why this matters:** the *value* `V(s)` of a state is the expected discounted return from there. Because the floor is random, we can't read it off one run. But here the hallway is small enough to **solve exactly** with a linear equation, and then check what averaging real episodes gives.

In [7]:
GAMMA, P_MOVE, GOAL = 0.9, 0.8, 4

def step(s, rng):
    moved = rng.random() < P_MOVE                       # WHAT: the randomness: with probability 0.8 the move works
    s2 = s + 1 if moved else s
    return s2, -0.04 + (1.0 if s2 == GOAL else 0.0)     # WHAT: -0.04 per step, +1 on reaching the goal

def run_episode(rng, start=0, cap=1000):
    s, steps = start, []
    while s != GOAL and len(steps) < cap:
        s2, r = step(s, rng)
        steps.append((s, r, s2)); s = s2
    return steps

def discounted_return(steps):
    return sum(r * GAMMA ** t for t, (_, r, _) in enumerate(steps))     # WHAT: r0 + g*r1 + g^2*r2 + ...  (the "return" from the start)

def exact_values(p_move=P_MOVE):
    # WHAT: V(s) = p*(r_move + g*V(s+1)) + (1-p)*(r_stay + g*V(s)), for s = 0..3 and V(4) = 0. That is 4 equations, 4 unknowns.
    # WHY:  "the value of here = the expected reward of the next step + gamma x the expected value of where I land"
    A = np.zeros((4, 4)); b = np.zeros(4)
    for s in range(4):
        r_move = -0.04 + (1.0 if s + 1 == GOAL else 0.0); r_stay = -0.04
        A[s, s] += 1 - (1 - p_move) * GAMMA                      # the "stay" branch refers back to V(s) itself
        if s + 1 < GOAL: A[s, s + 1] -= p_move * GAMMA            # the "move" branch refers to V(s+1)
        b[s] = p_move * r_move + (1 - p_move) * r_stay
    return np.linalg.solve(A, b)

V_exact = exact_values()
print("exact values V(0..3):", np.round(V_exact, 4))
print()
rng = np.random.default_rng(4)
print(" episodes | average return from cell 0 | exact V(0) | error | standard error of the average")
for n in (10, 100, 1000, 10_000):
    rets = np.array([discounted_return(run_episode(rng)) for _ in range(n)])
    print(f" {n:7d}  |         {rets.mean():8.4f}          |  {V_exact[0]:8.4f}  | {rets.mean() - V_exact[0]:+.4f} |        {rets.std(ddof=1) / np.sqrt(n):.4f}")

exact values V(0..3): [0.4982 0.6229 0.765  0.9268]

 episodes | average return from cell 0 | exact V(0) | error | standard error of the average
      10  |           0.4415          |    0.4982  | -0.0567 |        0.0270
     100  |           0.4835          |    0.4982  | -0.0147 |        0.0094
    1000  |           0.4985          |    0.4982  | +0.0003 |        0.0031
   10000  |           0.4983          |    0.4982  | +0.0001 |        0.0010


**What to notice.**
- **The average of real returns converges to the exact expected value.** The "standard error" column (the sigma-over-root-n law from Part 3, estimated from the data itself) says how far off to expect it to be: each error above is within about two standard errors. Averaging episode returns is called **Monte Carlo** value estimation. It's the simplest way to learn `V` or `Q` from experience, and it needs nothing but running episodes.
- Notebook 8 will compare this with the Q-learning update (which *bootstraps*: it updates from its own neighboring estimates after every step instead of waiting for the episode to end) and with your walk-backwards idea. Now you know what all of them are trying to estimate, and what the noise they fight looks like.

Now the other side: estimate the **transition probability itself** by counting what happened. Take the action "right" from random cells and count how often the move worked.

In [8]:
rng = np.random.default_rng(5)
print("  tries  | estimated P(move works) | true | error | expected error sqrt(p(1-p)/n)")
for n in (20, 200, 2000, 20_000):
    cells = rng.integers(0, 4, n)
    worked = np.array([step(c, rng)[0] == c + 1 for c in cells])      # WHAT: did the move succeed? (counted from experience)
    p_hat = worked.mean()                                             # WHAT: estimate = successes / tries
    print(f" {n:6d}  |         {p_hat:.3f}           | {P_MOVE}  | {p_hat - P_MOVE:+.3f} |     {np.sqrt(P_MOVE * (1 - P_MOVE) / n):.3f}")

  tries  | estimated P(move works) | true | error | expected error sqrt(p(1-p)/n)
     20  |         0.650           | 0.8  | -0.150 |     0.089
    200  |         0.810           | 0.8  | +0.010 |     0.028
   2000  |         0.811           | 0.8  | +0.011 |     0.009
  20000  |         0.804           | 0.8  | +0.004 |     0.003


**What to notice.** Same square-root law again: the *expected* error (right column) falls 10 times for 100 times more tries. A single row's actual error can land above or below that by luck, as the 2,000-try row did. **A model-based method** (learn `P` by counting, then solve) and a **model-free method** (average returns, or Q-learning) are two ways of using the same noisy experience. You'll race them in this notebook's sandbox.

# Part 5: Why a baseline helps (variance in policy-gradient learning)

> **You are here:** REINFORCE nudges the policy toward actions with **good return**. But returns are noisy, so each nudge is noisy. Here is the noise measured, and the standard trick that tames it.

Smallest possible case: one decision, two actions. The policy picks action 1 with probability `p = sigmoid(z)` (Notebook 2's neuron). Action 0 pays an average of 8, action 1 pays an average of 10, each with noise (standard deviation 1). Truthfully, pushing `z` up should help, and the true slope of the expected reward with respect to `z` is `p(1-p) * (10 - 8)`; with `p = 0.5` that's 0.5.

REINFORCE's single-sample estimate of that slope is `r * (a - p)`: the reward times how surprising the action was (Notebook 2, Part 9). Three versions:
- **plain:** `r * (a - p)`.
- **with a baseline:** `(r - b) * (a - p)` where `b` is the *average reward* (9). This is the "advantage" idea: how much better than usual.
- **a bad baseline that peeks at the action taken:** `(r - mean reward of the action taken) * (a - p)`.

In [9]:
rng = np.random.default_rng(6)
N = 200_000
p = 0.5
mu = np.array([8.0, 10.0])                                       # WHAT: average reward of action 0 and action 1
a = (rng.random(N) < p).astype(int)                              # WHAT: sample an action from the policy
r = mu[a] + rng.normal(0, 1.0, N)                                # WHAT: the noisy reward

g_plain = r * (a - p)                                            # WHAT: plain REINFORCE estimate of the slope
b = p * mu[1] + (1 - p) * mu[0]                                  # WHAT: baseline = the average reward under the policy (9.0)
g_base = (r - b) * (a - p)                                       # WHAT: advantage-style estimate
g_bad = (r - mu[a]) * (a - p)                                    # WHAT: baseline that depends on WHICH action was taken (not allowed)

# the true slope, by nudging the exact expected reward  J(z) = sigmoid(z)*10 + (1 - sigmoid(z))*8
J = lambda z: 1 / (1 + np.exp(-z)) * mu[1] + (1 - 1 / (1 + np.exp(-z))) * mu[0]
true_slope = (J(1e-5) - J(-1e-5)) / 2e-5
print(f"true slope of expected reward at p = 0.5 (by nudging): {true_slope:.4f}")
print()
print(f"{'estimate':38s}{'average':>10s}{'std of one sample':>20s}{'samples to get error 0.01':>34s}")
for name, g, ok in (("plain  r*(a-p)", g_plain, True), ("baseline  (r-9)*(a-p)", g_base, True), ("bad baseline  (r - mean of action)", g_bad, False)):
    need = f"{(g.std() / 0.01) ** 2:,.0f}" if ok else "n/a: converges to the wrong value"
    print(f"{name:38s}{g.mean():10.3f}{g.std():20.3f}{need:>34s}")

true slope of expected reward at p = 0.5 (by nudging): 0.5000

estimate                                 average   std of one sample         samples to get error 0.01
plain  r*(a-p)                             0.495               4.529                           205,141
baseline  (r-9)*(a-p)                      0.500               0.501                             2,515
bad baseline  (r - mean of action)         0.000               0.501 n/a: converges to the wrong value


In [10]:
fig, ax = plt.subplots(figsize=(7.6, 4.0))
bins = np.linspace(-7, 7, 113)
ax.hist(g_plain, bins=bins, histtype="step", color=ORANGE, linewidth=1.8, density=True)
ax.hist(g_base, bins=bins, histtype="step", color=BLUE, linewidth=1.8, density=True)
ax.axvline(0.5, color=INK, linewidth=1.2, linestyle="--")
ax.set_xlabel("one-sample estimate of the slope"); ax.set_ylabel("density")
tag(ax, -4.0, 0.43, "plain: action 0 samples", ORANGE, dx=0, ha="center"); tag(ax, 5.0, 0.43, "plain: action 1 samples", ORANGE, dx=0, ha="center")
tag(ax, 1.1, 0.62, "with baseline: one narrow hump", BLUE, dx=4)
tag(ax, 0.5, 0.82, "true slope 0.5", INK, dx=6)
ax.set_ylim(0, 0.9)
ax.set_title("Same average (dashed), very different single samples")
plt.show()

<Figure size 836x440 with 1 Axes>

**What to notice.**
- **Both honest versions average to the true slope (0.5).** Subtracting the baseline did not change where learning is heading. But the plain estimate's standard deviation is about **4.5** and the baseline version's about **0.5**: a **9 times smaller spread**, which by the square-root law means about **80 times fewer samples** (205,141 vs 2,515 in the table) for the same precision.
- Look at the picture: the plain estimate is **two humps**, one for each action. Every sample where action 1 was taken says "push up, about +5", and every sample where action 0 was taken says "push down, about -4". The true answer, +0.5, is the *tiny imbalance* between those two big opposing shouts, so it takes enormous numbers of samples to see. The baseline removes the big constant (about 9) from the reward, leaving only how much better or worse the outcome was than usual.
- **The bad baseline** (one that knows the action taken) averages to **0.000**, not 0.5. It has subtracted away the very signal we want. **A baseline may depend on the state, but never on the action you took.** That is the rule behind A2C: the critic `V(s)` is a state-only baseline, and `reward - V(s)` is the **advantage**.

# Part 6: Entropy (how random is a policy?)

> **You are here:** A2C had an `entropy 0.01` setting. **Entropy** measures how spread out a distribution is: `H = -sum( p * log(p) )`.

A policy that always picks one action has entropy 0 (no randomness). A policy that picks all `k` actions equally has the maximum, `log(k)`. Compute a few:

In [11]:
def entropy(p):
    p = np.asarray(p, float)
    p = p[p > 0]                                    # WHAT: skip zero-probability actions (0 * log 0 counts as 0)
    return float(max(0.0, -np.sum(p * np.log(p))))  # WHAT: H = -sum( p * log p )  (max(0, .) just avoids printing -0.000)

print("   policy (probabilities)         | entropy | max possible for that many actions")
for pol in ([1.0, 0.0], [0.9, 0.1], [0.5, 0.5], [0.97, 0.01, 0.01, 0.01], [0.25, 0.25, 0.25, 0.25]):
    print(f"   {str(pol):30s} | {entropy(pol):6.3f}  | {np.log(len(pol)):6.3f}")

   policy (probabilities)         | entropy | max possible for that many actions
   [1.0, 0.0]                     |  0.000  |  0.693
   [0.9, 0.1]                     |  0.325  |  0.693
   [0.5, 0.5]                     |  0.693  |  0.693
   [0.97, 0.01, 0.01, 0.01]       |  0.168  |  1.386
   [0.25, 0.25, 0.25, 0.25]       |  1.386  |  1.386


**What to notice.** Certainty is entropy 0; a coin flip is `log 2 = 0.693`; four equal choices are `log 4 = 1.386`. A policy that has "collapsed" onto one action has entropy near 0.

**An entropy bonus** adds `beta * H` to what the policy tries to maximize, with `beta` the knob (0.01 in the A2C notebook). It pushes the policy *toward* randomness, so it can't lock onto an action before it has tried the others. Check the slope of entropy with respect to the policy's logits (the numbers a softmax turns into probabilities) by formula and by nudging, as always:

In [12]:
def softmax(z):
    e = np.exp(z - np.max(z)); return e / e.sum()

def entropy_of_logits(z):
    return entropy(softmax(z))

z = np.array([2.0, 0.0, -1.0])
p_ = softmax(z); H = entropy(p_)
formula = -p_ * (np.log(p_) + H)                                   # WHAT: d(entropy)/d(logit_i) = -p_i * (log p_i + H)
nudged = np.array([(entropy_of_logits(z + e * 1e-6) - entropy_of_logits(z - e * 1e-6)) / 2e-6 for e in np.eye(3)])
print("probabilities:", np.round(p_, 3), "  entropy:", round(H, 3))
print("slope by formula:", np.round(formula, 5))
print("slope by nudging:", np.round(nudged, 5))
print()
z2 = z + 1.0 * formula                                             # WHAT: one gradient-ASCENT step on entropy (we want MORE randomness)
print(f"after one ascent step on entropy: probabilities {np.round(softmax(z2), 3)}, entropy {entropy_of_logits(z2):.3f} (was {H:.3f})")

probabilities: [0.844 0.114 0.042]   entropy: 0.524
slope by formula: [-0.29906  0.18792  0.11114]
slope by nudging: [-0.29906  0.18792  0.11114]

after one ascent step on entropy: probabilities [0.772 0.17  0.058], entropy 0.666 (was 0.524)


**What to notice.** The slopes match. The step moves probability *away* from the favorite and toward the rare actions, raising entropy. In the A2C loss that nudge competes with the "do what earned reward" nudge, and `beta` sets who wins: too big and the policy stays random forever; zero and it can collapse early. (How `beta`, and exploration in general, plays out on actual learning problems is Notebook 6, bandits. I haven't tested it here.)

# Part 7: Gamma is a survival probability

> **You are here:** `gamma` was introduced as "how much future rewards count". There's an equivalent reading in probability terms that makes the knob concrete.

Suppose you earn +1 every step, and after each step the game **continues with probability gamma** (it ends with probability `1 - gamma`). Then the *expected total reward* equals the discounted sum `1 + gamma + gamma^2 + ... = 1 / (1 - gamma)`. So discounting by `gamma` is the same as "an agent that expects the game might end at any step". Check by simulation:

In [13]:
rng = np.random.default_rng(7)
print(" gamma | exact discounted sum 1/(1-gamma) | simulated average total reward with a random stop | average episode length")
for g in (0.5, 0.9, 0.99):
    totals = []
    for _ in range(50_000):
        t = 1                                         # WHAT: the first step's +1 always happens
        while rng.random() < g:                       # WHAT: after each step, keep playing with probability gamma
            t += 1
        totals.append(t)                              # WHAT: one reward per step played = total reward = episode length
    print(f" {g:5} |             {1 / (1 - g):8.2f}              |                    {np.mean(totals):8.2f}                       |        {np.mean(totals):8.2f}")

 gamma | exact discounted sum 1/(1-gamma) | simulated average total reward with a random stop | average episode length


   0.5 |                 2.00              |                        1.99                       |            1.99


   0.9 |                10.00              |                       10.09                       |           10.09


  0.99 |               100.00              |                      100.15                       |          100.15


**What to notice.** The two numbers agree to within about two standard errors (50,000 simulated games each; the standard error is about 0.04 for gamma 0.9, and the simulation says 10.09 against an exact 10.00). **Gamma = 0.9 means "plan for roughly 10 steps ahead"; gamma = 0.99 means roughly 100.** A cleaner way to *choose* gamma than guessing: ask how many steps ahead the decisions you care about reach (a CartPole balance failure is a few dozen steps away, a long maze path more). Larger gamma also means noisier returns, since more random steps are summed (Part 3).

# Part 8: How many seeds before "A beat B" means anything?

> **You are here:** every RL experiment is random: different seeds, different results. You'll compare algorithms constantly, so know how much evidence a handful of runs gives.

Pretend algorithm **A** scores 100 on average and **B** scores 105 (B is truly 5 points better), but each *run* has noise with a standard deviation of 20 (the run-to-run scatter RL really has). You run each algorithm with `n` seeds and compare the averages. How often is B's average *higher*? How often does a simple test (difference divided by its standard error beyond 2) say "clearly different"? And, importantly, how often does that test cry wolf when the algorithms are **identical**?

In [14]:
rng = np.random.default_rng(8)
def experiment(n, mean_b, trials=4000):
    A = rng.normal(100, 20, (trials, n)); B = rng.normal(mean_b, 20, (trials, n))
    diff = B.mean(1) - A.mean(1)
    se = np.sqrt(A.var(1, ddof=1) / n + B.var(1, ddof=1) / n)       # WHAT: standard error of the difference, estimated from the runs themselves
    return np.mean(diff > 0), np.mean(diff / se > 2), np.mean(np.abs(diff / se) > 2)

seed_counts = [3, 5, 10, 30, 100, 300]
seed_rows = []
print(" seeds per algorithm | B's average is higher | test says 'B clearly better' | test cries wolf when A = B")
for n in seed_counts:
    higher, detect, _ = experiment(n, 105)
    _, _, false_alarm = experiment(n, 100)
    seed_rows.append((n, higher, detect, false_alarm))
    print(f"        {n:4d}         |        {higher:6.1%}         |           {detect:6.1%}           |          {false_alarm:6.1%}")

 seeds per algorithm | B's average is higher | test says 'B clearly better' | test cries wolf when A = B
           3         |         63.5%         |            10.2%           |           11.3%
           5         |         64.8%         |             8.9%           |            8.1%
          10         |         70.5%         |             7.7%           |            6.1%
          30         |         83.2%         |            15.9%           |            5.5%
         100         |         96.4%         |            40.2%           |            4.5%
         300         |        100.0%         |            85.2%           |            5.0%


In [15]:
fig, ax = plt.subplots(figsize=(7.6, 4.0))
ns_ = [r[0] for r in seed_rows]
ax.plot(ns_, [r[1] for r in seed_rows], color=BLUE, marker="o", markersize=4)
ax.plot(ns_, [r[2] for r in seed_rows], color=ORANGE, marker="o", markersize=4)
ax.axhline(0.8, color="#898781", linestyle="--", linewidth=1.1)
ax.set_xscale("log"); ax.set_ylim(0, 1.02); ax.set_xlabel("seeds per algorithm (log scale)"); ax.set_ylabel("fraction of experiments")
tag(ax, ns_[0], seed_rows[0][1], "B's average is higher", BLUE, dx=0, dy=12)
tag(ax, ns_[0], seed_rows[0][2], "test says clearly better", ORANGE, dx=0, dy=12)
tag(ax, ns_[-2], 0.8, "80% line", MUTED, dx=0, dy=-9)
ax.set_title("A real 5-point improvement (noise sd 20) needs many seeds to be reliably seen")
plt.show()

<Figure size 836x440 with 1 Axes>

**What to notice.**
- With 3 to 5 seeds each, a **real** 5-point improvement shows up as "B's average is higher" only about 64% to 65% of the time (a coin flip is 50%), and the test calls it "clearly better" only about 9% to 10% of the time. At 30 seeds B's average is higher 83% of the time, but the test still sees it only 16% of the time. Even **100 seeds** gives just 40% detection; it takes about **300 seeds per algorithm** (85%) here to reliably see a 5-point gain against noise of 20.
- Look at the last column in the table. With 3 seeds the test cries wolf **11%** of the time when the algorithms are identical (twice the 5% it's meant to), because with 3 numbers the estimated standard error is itself unreliable. By 30 seeds it is back near 5%. So tiny experiments fail both ways: they miss real effects and sometimes invent fake ones.
- This matches Notebook 4's lesson that judging a design from one run misleads. **Practical rule:** report the mean and the standard error across several seeds, and treat differences smaller than about 2 standard errors as "not shown". (Assumptions here: normal noise, equal noise in both, independent seeds, a made-up effect size. Real RL curves are often messier.)

# Part 9: The vocabulary, now computable

> **You are here:** the RL words you've been reading, each tied to something you computed in this notebook.

| RL word | Probability meaning | Where you computed it |
|---|---|---|
| **policy** | a distribution over actions, per state | Part 1 |
| **return** | a random number (sum of discounted rewards) | Part 4 |
| **Q-value / V-value** | the expected value of the return | Parts 2 and 4 |
| **exploration noise** | sampling from the policy | Part 1 |
| **number of episodes needed** | set by the variance of the return and the square-root law | Part 3 |
| **advantage** | (return) minus (a baseline that doesn't peek at the action) | Part 5 |
| **entropy bonus** | reward for keeping the policy spread out | Part 6 |
| **gamma** | survival probability / horizon | Part 7 |
| **seeds** | independent repeats; judge with mean and standard error | Part 8 |
| **transition model** | `P(next state | state, action)`, estimable by counting | Part 4 |

# Part 10: Sandbox (your ideas compete here)

> **You are here:** estimate the value of the start of the slippery hallway from a **budget of 200 episodes**. This is the ruler for the three families of method you'll meet next.

**The ruler.** An estimator is a function `estimator(episodes)` that returns its best guess of `V(0)`, using **only** the 200 episodes it's given (each is a list of `(state, reward, next_state)` steps). The exact answer is known, so each estimator is scored over **100 repeated experiments** by its **bias** (average error), **spread** (standard deviation of its guesses) and **RMSE** (typical total error, combining both). Reference entries:
- **Monte Carlo:** average the discounted return of each episode (Part 4).
- **Model-based:** count how often moves worked (Part 4), build the transition model, and solve the exact equations. It is told the reward rules; only the slip chance is unknown.
- **TD(0):** after every single step, nudge `V(s)` toward `reward + gamma * V(next)` by a step size `alpha`. This is the *bootstrapping* idea behind Q-learning. (Gradient step on a squared error: Notebook 1, Part 8.)

In [16]:
N_EPISODES, N_REPEATS = 200, 100
repeat_sets = []                                                      # WHAT: 100 experiments, each a set of 200 episodes (the SAME sets for every estimator, so the comparison is fair)
for rep in range(N_REPEATS):
    r_ = np.random.default_rng(10_000 + rep)
    repeat_sets.append([run_episode(r_) for _ in range(N_EPISODES)])

def est_monte_carlo(episodes):
    return float(np.mean([discounted_return(ep) for ep in episodes]))

def est_model_based(episodes):
    tries = moves = 0
    for ep in episodes:
        for s, _, s2 in ep:
            tries += 1; moves += (s2 == s + 1)                        # WHAT: count how often a move actually worked
    return float(exact_values(moves / max(tries, 1))[0])               # WHAT: plug the counted chance into the exact equations

def td0(alpha):
    def est(episodes):
        V = np.zeros(GOAL + 1)                                        # V[GOAL] stays 0: the game is over there
        for ep in episodes:
            for s, r, s2 in ep:
                V[s] += alpha * (r + GAMMA * V[s2] - V[s])            # WHAT: nudge V(s) toward "reward + gamma x my current guess for the next state"
        return float(V[0])
    return est

def judge(estimator):
    guesses = np.array([estimator(eps) for eps in repeat_sets])
    bias = guesses.mean() - V_exact[0]
    return bias, guesses.std(), float(np.sqrt(np.mean((guesses - V_exact[0]) ** 2)))

def race5(entries):
    print(f"exact V(0) = {V_exact[0]:.4f}      (each row: 100 experiments x 200 episodes)")
    print(f"{'estimator':34s}{'bias':>10s}{'spread':>10s}{'RMSE':>10s}")
    for label, est in entries:
        bias, spread, rmse = judge(est)
        print(f"{label:34s}{bias:+10.4f}{spread:10.4f}{rmse:10.4f}")

reference5 = [("Monte Carlo (average of returns)", est_monte_carlo),
              ("model-based (count, then solve)", est_model_based),
              ("TD(0), alpha = 0.02", td0(0.02)),
              ("TD(0), alpha = 0.1", td0(0.1)),
              ("TD(0), alpha = 0.5", td0(0.5))]
race5(reference5)

exact V(0) = 0.4982      (each row: 100 experiments x 200 episodes)
estimator                               bias    spread      RMSE
Monte Carlo (average of returns)     -0.0005    0.0075    0.0075
model-based (count, then solve)      -0.0004    0.0076    0.0076
TD(0), alpha = 0.02                  -0.2452    0.0047    0.2452
TD(0), alpha = 0.1                   +0.0085    0.0179    0.0198
TD(0), alpha = 0.5                   +0.0446    0.0229    0.0501


**What to notice.**
- **Monte Carlo and model-based tie** (RMSE 0.0075 vs 0.0076, both essentially unbiased) on this problem. I wouldn't generalize from one tiny problem where the model is easy to learn and the reward rules were handed to the model-based method.
- **TD(0) with a small step (alpha = 0.02)** has the *lowest spread* (0.0047) and the *worst error* (RMSE 0.245): after 200 episodes it is still stuck near its all-zero starting guess (bias -0.245). A tight spread does not mean accurate: this is the learning-rate window from Notebook 1 again (too small is slow).
- **TD(0) with alpha = 0.1** is the best TD setting I tried (RMSE 0.020, about 2.6 times worse than Monte Carlo here). **alpha = 0.5** is noisier (spread 0.023) and biased upward (+0.045), because with a big step the last few updates dominate.
- What this ruler does **not** measure: TD's real selling points, which are learning *during* an episode and working when episodes never end. Only three alphas were tried and only one problem, so don't read this as "Monte Carlo beats TD".

### Your slot
Edit `my_estimator` and re-run. Ideas: average two of the estimators; TD(0) with `alpha` that shrinks over time; run TD(0) over the episodes **in reverse order** (and then your walk-backwards idea, applied inside each episode: update the last step first, so the goal's reward travels back in one pass); anything that uses the 200 episodes only. Rule: you may not look at `V_exact` inside the estimator.

In [17]:
def my_estimator(episodes):
    # SANDBOX: replace with your own idea. Input: 200 episodes of (s, r, s2) steps. Return a guess for V(0).
    return est_monte_carlo(episodes)           # <- starts as plain Monte Carlo

race5(reference5 + [("MY ESTIMATOR", my_estimator)])

exact V(0) = 0.4982      (each row: 100 experiments x 200 episodes)
estimator                               bias    spread      RMSE
Monte Carlo (average of returns)     -0.0005    0.0075    0.0075
model-based (count, then solve)      -0.0004    0.0076    0.0076


TD(0), alpha = 0.02                  -0.2452    0.0047    0.2452
TD(0), alpha = 0.1                   +0.0085    0.0179    0.0198


TD(0), alpha = 0.5                   +0.0446    0.0229    0.0501
MY ESTIMATOR                         -0.0005    0.0075    0.0075


# Part 11: The knob ledger, self-check, and what's next

## Knob ledger (Notebooks 1 to 5)
Families: **objective**, **model**, **search**, **experience**, **judging**. New in this notebook: how much experience you collect, the baseline, the entropy weight, gamma, and the number of seeds.

In [18]:
mc_b, mc_s, mc_r = judge(est_monte_carlo)
mb_b, mb_s, mb_r = judge(est_model_based)
rows = [
    ("learning rate (NB1)", "search", "too small: slow", "too big: blows up"),
    ("model size / leash / early stop (NB3)", "model, objective, judging", "too small: underfits", "too big, unleashed: overfits"),
    ("width, depth, activation, init (NB4)", "model, search", "1 hidden neuron can't solve XOR", "wide enough solves it from most starts"),
    ("episodes / samples collected", "experience", f"10 samples: spread of the average {spreads[10]:.2f}", f"10,000 samples: {spreads[10_000]:.2f} (100x data, 10x smaller)"),
    ("baseline in policy gradient", "objective", f"none: single-sample spread {g_plain.std():.1f}", f"state-only baseline: {g_base.std():.1f} (a baseline that sees the action: biased to {g_bad.mean():.2f})"),
    ("entropy weight beta", "objective", "0: policy can collapse early", "too big: stays random (not tested here; Notebook 6)"),
    ("gamma", "objective", "0.5: plans about 2 steps ahead", "0.99: plans about 100 steps ahead, much noisier returns"),
    ("number of seeds", "judging", f"3 seeds: test cries wolf {seed_rows[0][3]:.0%} when A = B; detects a real gain {seed_rows[0][2]:.0%}", f"300 seeds: cries wolf {seed_rows[-1][3]:.0%}; detects {seed_rows[-1][2]:.0%}"),
    ("estimator choice for V", "search", f"Monte Carlo RMSE {mc_r:.3f}", f"model-based RMSE {mb_r:.3f}"),
]
print(f"{'knob':40s}{'family':26s}{'at one extreme':74s}{'at the other'}")
print("-" * 205)
for knob, fam, a, b in rows:
    print(f"{knob:40s}{fam:26s}{a:74s}{b}")

knob                                    family                    at one extreme                                                            at the other
-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
learning rate (NB1)                     search                    too small: slow                                                           too big: blows up
model size / leash / early stop (NB3)   model, objective, judging too small: underfits                                                      too big, unleashed: overfits
width, depth, activation, init (NB4)    model, search             1 hidden neuron can't solve XOR                                           wide enough solves it from most starts
episodes / samples collected            experience                10 samples: spread of the average 4.33                               

## Self-check (answer in your own words before you read the answers)
| Question | Short answer |
|---|---|
| What is a policy, in probability terms? | A distribution over actions for each state; you act by sampling from it. |
| Why is a Q-value an expected value? | The return after (s, a) is random; Q is its long-run average, `E[return]`. |
| Why does RL need so many episodes? | Each return is one noisy sample; the error of an average shrinks only like 1/sqrt(n). |
| What does a baseline do, and what must it NOT do? | It cuts the variance of the gradient estimate without changing its average. It must not depend on the action taken, or it cancels the signal. |
| What does entropy measure? | How spread out the policy is: 0 for always the same action, `log k` for uniform over `k`. |
| What does gamma mean as a probability? | The chance the game continues each step. The planning horizon is about `1/(1-gamma)` steps. |
| How many seeds can you trust? | Few seeds miss real effects and invent fake ones; report mean and standard error, and don't claim differences under about 2 standard errors. |

## What's next
- You can now read the quantities in REINFORCE and A2C as expectations, variances and distributions, and you know why they're noisy.
- You have a ruler (Part 10) that compares **model-free**, **model-based** and **bootstrapped** value estimation on a problem you can solve exactly.

**Notebook 6: bandits.** The simplest RL problem: one state, several actions, random rewards. All the difficulty of *explore versus exploit* with none of the rest. You'll race epsilon-greedy, optimism, UCB, Thompson sampling and a softmax policy with entropy on a shared ruler, and test the entropy and baseline ideas from this notebook on actual learning.